In [3]:
# just to speed up things a little bit by running two instances of the R interpretor

In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [2]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("nlme")
    library("corHMM")
    library("geiger")
    library("OUwie")
    library("reshape2")
    library("ggplot2")
})

stopifnot(packageVersion("OUwie") == "2.16")
stopifnot(packageVersion("corHMM") == "2.8")

In [3]:
STATES <- read.csv("../../data/chapter2/FREDv3subset/finalized_states_395_species.csv", stringsAsFactors = TRUE)[, c("binominal", "state")] # finalized mycorrhizal states
COLLAB_AXIS <- read.csv("../../data/chapter2/FREDv3subset/collab_ord1_species_avgs_SRL_RD.csv", stringsAsFactors = TRUE) # first order species averaged RD and SRL values
MERGED <- merge(x = STATES, y = COLLAB_AXIS, by = "binominal")
stopifnot(nrow(MERGED)==395)

PHYLOGENY <- ape::multi2di(ape::read.tree("../../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre")) # phylogenetic tree created for the 395 species using U.PhyloMaker
stopifnot(length(PHYLOGENY$tip.label)==395)

In [7]:
data <- data.frame(binominal = gsub(MERGED$binominal, pattern = ' ', replacement = '_'), RD = MERGED$F00679, SRL = MERGED$F00727, myco = gsub(x = MERGED$state, pattern = '/', replacement = '')) # MERGED contains '/'
matched_row_indices <- match(PHYLOGENY$tip.label, data$binominal)
stopifnot(all(data$binominal[matched_row_indices] == PHYLOGENY$tip.label))

data <- data[matched_row_indices, ]
rownames(data) <- NULL
stopifnot(all(data$binominal == PHYLOGENY$tip.label))
stopifnot(length(unique(data$binominal)) == length(data$binominal))

In [8]:
SRLdata <- data[, c("binominal", "myco", "SRL")] # for specific root length

In [7]:
unique(SRLdata$myco)

[1] "AMNM"  "AM"    "ErM"   "NM"    "AMEcM" "EcM"

In [8]:
min(MERGED$F00679)
max(MERGED$F00679)

[1] 0.05103318

[1] 12.73

In [9]:
min(MERGED$F00727)
max(MERGED$F00727)

[1] 0.0493

[1] 840.0963

In [10]:
tm <- Sys.time()

# rate.cat = 1 and null.model = FALSE 

# moving the SYM variants of the CD models first as this is where the exception was thrown earlier - so we don't have to wait hours to get the exception
SYM_OUM_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = FALSE) # AN EXCEPTION IS THROWN HERE?????
SYM_OUMA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
SYM_OUMV_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
SYM_OUMVA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ER_OUM_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = FALSE) # NOW IT HAPPENS HERE????
ER_OUMA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ER_OUMV_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ER_OUMVA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ARD_OUM_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ARD_OUMA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ARD_OUMV_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ARD_OUMVA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

# rate.cat = 2 and null.model = TRUE

ER_OUM_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ER_OUMA_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = TRUE) # now the error happened below??????
ER_OUMV_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ER_OUMVA_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

ARD_OUM_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ARD_OUMA_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ARD_OUMV_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ARD_OUMVA_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

SYM_OUM_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = TRUE)
SYM_OUMA_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
SYM_OUMV_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
SYM_OUMVA_SRL_CID <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

tm <- Sys.time() - tm

save(ER_OUM_SRL_CD, ER_OUMA_SRL_CD, ER_OUMV_SRL_CD, ER_OUMVA_SRL_CD, ARD_OUM_SRL_CD, ARD_OUMA_SRL_CD, ARD_OUMV_SRL_CD, ARD_OUMVA_SRL_CD, SYM_OUM_SRL_CD, SYM_OUMA_SRL_CD, SYM_OUMV_SRL_CD, SYM_OUMVA_SRL_CD, file = "../rdata/OU_SRL_CD.RData")
save(ER_OUM_SRL_CID, ER_OUMA_SRL_CID, ER_OUMV_SRL_CID, ER_OUMVA_SRL_CID, ARD_OUM_SRL_CID, ARD_OUMA_SRL_CID, ARD_OUMV_SRL_CID, ARD_OUMVA_SRL_CID, SYM_OUM_SRL_CID, SYM_OUMA_SRL_CID, SYM_OUMV_SRL_CID, SYM_OUMVA_SRL_CID, file = "../rdata/OU_SRL_CID.RData")

Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


...

Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLdata, rate.cat = 2, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


ERROR: Error in liks_houwie$TotalLik: $ operator is invalid for atomic vectors


In [15]:
ER_OUMA_SRL_CID

ERROR: Error: object 'ER_OUMA_SRL_CID' not found


## ___Numerically encoded states___
------------------------------

In [20]:
STATE_NUMBERS <- c("AMNM" = 1, "AM" = 2, "ErM" = 3, "NM" = 4, "AMEcM" = 5, "EcM" = 6)

In [25]:
SRLstatenumeric <- SRLdata
SRLstatenumeric$myco <- unname(STATE_NUMBERS[SRLstatenumeric$myco])

In [27]:
head(SRLstatenumeric)

,binominal,myco,SRL
,<chr>,<dbl>,<dbl>
1,Anaphalis_aureopunctata,1,479.1550
2,Anaphalis_hancockii,2,319.8579
3,Solidago_decurrens,2,202.8837
4,Doellingeria_scabra,2,219.2008
5,Aster_tataricus,2,297.1100
6,Artemisia_igniaria,2,204.1529


In [ ]:
# try and see if the same error happens with states encoded as numbers

SYM_OUM_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = FALSE) # AN EXCEPTION IS THROWN HERE????? (1st run)
SYM_OUMA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
SYM_OUMV_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
SYM_OUMVA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ER_OUM_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = FALSE) # NOW IT HAPPENS HERE???? (2nd run)
ER_OUMA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ER_OUMV_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ER_OUMVA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ARD_OUM_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ARD_OUMA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ARD_OUMV_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ARD_OUMVA_SRL_CD <- OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


...

Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = PHYLOGENY, data = SRLstatenumeric, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


## ___Model fitting with state customizations___
-----------------------------------------

In [12]:
state_altered <- MERGED[-which(MERGED$state == "ErM"), c("binominal", "state", "F00727")] # drop the ErM records
state_altered$binominal <- gsub(state_altered$binominal, pattern = ' ', replacement = '_') # replace the spaces in the binominal name with underscores to match the phylogeny
state_altered$state <- gsub(state_altered$state, pattern = '/', replacement = '') # remove the forward slashes in the mycorrhizal state encodings
state_altered$state[state_altered$state == "AMNM"] = "NM" # update the AM/NM encoding to NM

In [13]:
table(state_altered$state) # only got 4 states


   AM AMEcM   EcM    NM 
  300    15    65    12 

In [14]:
# now remove all the ErM tips from the phylogeny
phylogeny <- ape::drop.tip(phy = PHYLOGENY, tip = setdiff(PHYLOGENY$tip.label, state_altered$binominal), trim.internal = TRUE)
phylogeny


Phylogenetic tree with 392 tips and 391 internal nodes.

Tip labels:
  Anaphalis_aureopunctata, Anaphalis_hancockii, Solidago_decurrens, Doellingeria_scabra, Aster_tataricus, Artemisia_igniaria, ...
Node labels:
  , Spermatophyta, Mesangiospermae, mrcaott2ott121, eudicotyledons, mrcaott2ott969, ...

Rooted; includes branch length(s).

In [16]:
# all(state_altered$binominal[match(phylogeny$tip.label, state_altered$binominal)] == phylogeny$tip.label)
state_altered <- state_altered[match(phylogeny$tip.label, state_altered$binominal), ] # reorder the dataset to match the tip label order in the phylogeny
stopifnot(all(state_altered$binominal == phylogeny$tip.label))

In [17]:
tm <- Sys.time()

# rate.cat = 1 and null.model = FALSE 

ER_OUM_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ER_OUMA_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ER_OUMV_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ER_OUMVA_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

ARD_OUM_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = FALSE)
ARD_OUMA_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
ARD_OUMV_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
ARD_OUMVA_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

SYM_OUM_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = FALSE)
SYM_OUMA_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = FALSE)
SYM_OUMV_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = FALSE)
SYM_OUMVA_SRL_CD <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = FALSE)

# rate.cat = 2 and null.model = TRUE

ER_OUM_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ER", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ER_OUMA_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ER_OUMV_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ER_OUMVA_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ER", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

ARD_OUM_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUM", nSim = 30, null.model = TRUE)
ARD_OUMA_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
ARD_OUMV_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
ARD_OUMVA_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "ARD", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

SYM_OUM_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUM", nSim = 30, null.model = TRUE)
SYM_OUMA_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMA", nSim = 30, null.model = TRUE)
SYM_OUMV_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMV", nSim = 30, null.model = TRUE)
SYM_OUMVA_SRL_CID <- OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, discrete_model = "SYM", continuous_model = "OUMVA", nSim = 30, null.model = TRUE)

tm <- Sys.time() - tm

save(ER_OUM_SRL_CD, ER_OUMA_SRL_CD, ER_OUMV_SRL_CD, ER_OUMVA_SRL_CD, ARD_OUM_SRL_CD, ARD_OUMA_SRL_CD, ARD_OUMV_SRL_CD, ARD_OUMVA_SRL_CD, SYM_OUM_SRL_CD, SYM_OUMA_SRL_CD, SYM_OUMV_SRL_CD, SYM_OUMVA_SRL_CD, file = "../rdata/OU_SRL_CD_4states.RData")
save(ER_OUM_SRL_CID, ER_OUMA_SRL_CID, ER_OUMV_SRL_CID, ER_OUMVA_SRL_CID, ARD_OUM_SRL_CID, ARD_OUMA_SRL_CID, ARD_OUMV_SRL_CID, ARD_OUMVA_SRL_CID, SYM_OUM_SRL_CID, SYM_OUMA_SRL_CID, SYM_OUMV_SRL_CID, SYM_OUMVA_SRL_CID, file = "../rdata/OU_SRL_CID_4states.RData")

Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


...

Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...


Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = phylogeny, data = state_altered, rate.cat = 2, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 30 simmaps using the nlopt_ln optimization protocol...
